# Сравнение ResNet-50 vs ViT-Small

Сравнительный анализ CNN и Transformer архитектур для Vehicle Re-ID

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Загрузка результатов

In [ ]:
# Загружаем результаты обучения
with open('./checkpoints_resnet50/results.json', 'r') as f:
    resnet_results = json.load(f)

with open('./checkpoints_vit/results.json', 'r') as f:
    vit_results = json.load(f)

print('Results loaded!')
print(f"ResNet-50: {resnet_results['model']}")
print(f"ViT: {vit_results['model']}")

## 2. Сравнение метрик

In [ ]:
# Извлекаем метрики
metrics = ['Rank-1', 'Rank-5', 'Rank-10', 'mAP']

resnet_metrics = [resnet_results['results'][m] for m in metrics]
vit_metrics = [vit_results['results'][m] for m in metrics]

# Таблица результатов
print('='*60)
print('СРАВНЕНИЕ РЕЗУЛЬТАТОВ НА VeRi-776')
print('='*60)
print(f'{"Метрика":<15} {"ResNet-50":<15} {"ViT-Small":<15} {"Разница":<15}')
print('-'*60)
for m, r, v in zip(metrics, resnet_metrics, vit_metrics):
    diff = v - r
    sign = '+' if diff > 0 else ''
    print(f'{m:<15} {r:>12.2f}% {v:>12.2f}% {sign}{diff:>12.2f}%')
print('='*60)

## 3. График сравнения метрик

In [ ]:
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, resnet_metrics, width, label='ResNet-50', color='#2ecc71')
bars2 = ax.bar(x + width/2, vit_metrics, width, label='ViT-Small', color='#3498db')

ax.set_ylabel('Accuracy (%)')
ax.set_title('ResNet-50 vs ViT-Small на VeRi-776')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 100)

# Добавляем значения над столбцами
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=10)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('./comparison_metrics.png', dpi=150)
plt.show()
print('Saved: comparison_metrics.png')

## 4. Кривые обучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1 = axes[0]
ax1.plot(resnet_results['history']['loss'], label='ResNet-50', color='#2ecc71', linewidth=2)
ax1.plot(vit_results['history']['loss'], label='ViT-Small', color='#3498db', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# mAP
ax2 = axes[1]
epochs_eval = list(range(10, len(resnet_results['history']['mAP'])*10+1, 10))
ax2.plot(epochs_eval, resnet_results['history']['mAP'], 'o-', label='ResNet-50', color='#2ecc71', linewidth=2)
ax2.plot(epochs_eval[:len(vit_results['history']['mAP'])], vit_results['history']['mAP'], 's-', label='ViT-Small', color='#3498db', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('mAP (%)')
ax2.set_title('Validation mAP')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./training_curves_comparison.png', dpi=150)
plt.show()
print('Saved: training_curves_comparison.png')

## 5. Сводная таблица для диплома

In [ ]:
print('\n' + '='*70)
print('ДАННЫЕ ДЛЯ ВСТАВКИ В ДИПЛОМ (Chapter3.tex)')
print('='*70)
print()
print('Таблица: Сравнение ResNet-50 и ViT-Small на VeRi-776')
print('-'*70)
print(f'ResNet-50 + BNNeck: Rank-1={resnet_results["results"]["Rank-1"]:.1f}%, '
      f'Rank-5={resnet_results["results"]["Rank-5"]:.1f}%, '
      f'mAP={resnet_results["results"]["mAP"]:.1f}%')
print(f'ViT-Small + BNNeck: Rank-1={vit_results["results"]["Rank-1"]:.1f}%, '
      f'Rank-5={vit_results["results"]["Rank-5"]:.1f}%, '
      f'mAP={vit_results["results"]["mAP"]:.1f}%')
print()
print('Скопируй эти значения в таблицы Chapter3.tex вместо XX.X')
print('='*70)

## 6. Выводы

In [ ]:
diff_map = vit_results['results']['mAP'] - resnet_results['results']['mAP']
diff_rank1 = vit_results['results']['Rank-1'] - resnet_results['results']['Rank-1']

winner_map = 'ViT-Small' if diff_map > 0 else 'ResNet-50'
winner_rank1 = 'ViT-Small' if diff_rank1 > 0 else 'ResNet-50'

print('ВЫВОДЫ:')
print(f'- По mAP лучше: {winner_map} (разница {abs(diff_map):.1f}%)')
print(f'- По Rank-1 лучше: {winner_rank1} (разница {abs(diff_rank1):.1f}%)')
print()
if diff_map > 0:
    print('ViT показывает лучшие результаты благодаря глобальному вниманию.')
else:
    print('ResNet показывает лучшие результаты благодаря эффективным свёрточным операциям.')